# Data Quality Profiling and Targeted Cleaning

This notebook profiles the raw landslide dataset, documents the requested quality checks, and creates a **separate** `cleaned_raw_data` table from `raw_data`. The source `raw_data` table stays unchanged.


## Exact findings from the raw CSV

- Rows profiled: **48,108**
- Columns profiled: **52**
- `Date`: **0 malformed values**, **892** blanks kept as `NULL`
- `Response_Time_Min`: **0 malformed numeric tokens**, **1555** blanks kept as `NULL`
- `Historical_Landslide_Count`: min **0**, max **12**, invalid rows **0**
- `Houses_Damaged`: min **0**, max **65**, invalid rows **0**
- `Bridges_Damaged`: min **0**, max **5**, invalid rows **0**
- `State`: **18** labels, no inconsistency after trim/case normalization
- `Soil_Type`: **7** labels, no inconsistency after trim/case normalization
- `Human_Resources_Deployed`: **95** integer-like levels stored as float-style text; clean as `INT`


In [ ]:
import csv
import os
from collections import defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pymysql
from dotenv import load_dotenv
from IPython.display import Markdown, display


In [ ]:
ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CSV_PATH = ROOT_DIR / 'data' / 'Raw_Data.csv'
RAW_TABLE = 'raw_data'
CLEAN_TABLE = 'cleaned_raw_data'
load_dotenv(ROOT_DIR / '.env')

COLUMN_SPECS = [('Event_ID', 'VARCHAR(20)'), ('Date', 'DATE'), ('State', 'VARCHAR(100)'), ('District', 'VARCHAR(100)'), ('Latitude', 'DECIMAL(10,6)'), ('Longitude', 'DECIMAL(10,6)'), ('Season', 'VARCHAR(30)'), ('Rainfall_mm', 'DECIMAL(10,2)'), ('Elevation_m', 'DECIMAL(10,2)'), ('Slope_Degree', 'DECIMAL(10,2)'), ('Soil_Type', 'VARCHAR(50)'), ('NDVI', 'DECIMAL(10,3)'), ('Temperature_C', 'DECIMAL(6,2)'), ('Humidity', 'DECIMAL(6,2)'), ('Distance_to_River_km', 'DECIMAL(10,2)'), ('Land_Use', 'VARCHAR(50)'), ('Historical_Landslide_Count', 'INT'), ('Landslide_Occurred', 'VARCHAR(10)'), ('Landslide_Risk', 'VARCHAR(20)'), ('Casualties', 'INT'), ('Injured', 'INT'), ('Missing_Persons', 'INT'), ('Families_Affected', 'INT'), ('Population_Affected', 'INT'), ('Houses_Damaged', 'INT'), ('Roads_Blocked_km', 'DECIMAL(10,2)'), ('Bridges_Damaged', 'INT'), ('Economic_Loss_INR', 'DECIMAL(15,2)'), ('Cropland_Damaged', 'DECIMAL(12,2)'), ('Livestock_Lost', 'INT'), ('Response_ID', 'VARCHAR(20)'), ('Response_Time_Min', 'DECIMAL(10,2)'), ('Rescue_Duration_Hours', 'DECIMAL(10,2)'), ('Human_Resources_Deployed', 'INT'), ('Rescue_Teams', 'INT'), ('NDRF_Teams', 'INT'), ('SDRF_Teams', 'INT'), ('Volunteers', 'INT'), ('Ambulances', 'INT'), ('Helicopters', 'INT'), ('Excavators', 'INT'), ('JCB_Machines', 'INT'), ('Cranes', 'INT'), ('Relief_Camps', 'INT'), ('Evacuated_People', 'INT'), ('Aid_Amount_INR', 'DECIMAL(15,2)'), ('Relief_Materials_Tons', 'DECIMAL(10,2)'), ('Compensation_Provided', 'VARCHAR(10)'), ('Recovery_Days', 'INT'), ('Power_Outage_Hours', 'DECIMAL(10,2)'), ('Water_Supply_Disrupted', 'VARCHAR(10)'), ('Communication_Disruption', 'VARCHAR(10)')]
TARGET_TYPE = dict(COLUMN_SPECS)
TARGETED_RANGE_COLUMNS = ['Historical_Landslide_Count', 'Houses_Damaged', 'Bridges_Damaged']


In [ ]:
def clean(value):
    if value is None:
        return None
    value = value.strip()
    return value or None

def parse_decimal(value):
    value = clean(value)
    if value is None:
        return None
    try:
        return Decimal(value)
    except InvalidOperation:
        return None

def is_date(value):
    value = clean(value)
    if value is None:
        return False
    try:
        datetime.strptime(value, '%Y-%m-%d')
        return True
    except ValueError:
        return False

def as_markdown_table(rows, headers):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    for row in rows:
        lines.append('| ' + ' | '.join(str(row.get(header, '')) for header in headers) + ' |')
    return '\n'.join(lines)


In [ ]:
with CSV_PATH.open('r', encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    headers = reader.fieldnames
    rows = list(reader)

summary_rows = []
for header in headers:
    values = [clean(row[header]) for row in rows]
    present = [value for value in values if value is not None]
    examples = []
    for value in present:
        if value not in examples:
            examples.append(value)
        if len(examples) == 3:
            break
    raw_profile = 'text'
    if present and all(is_date(value) for value in present):
        raw_profile = 'date_text'
    else:
        numeric_values = [parse_decimal(value) for value in present]
        if present and all(value is not None for value in numeric_values):
            raw_profile = 'numeric_text_integral' if all(value == value.to_integral_value() for value in numeric_values) else 'numeric_text_decimal'
    summary_rows.append({
        'Column': f'`{header}`',
        'Raw Profile': raw_profile,
        'Target Type': f'`{TARGET_TYPE[header]}`',
        'Non-Null': len(present),
        'Nulls': len(rows) - len(present),
        'Examples': ', '.join(examples) if examples else '—',
    })

display(Markdown(as_markdown_table(summary_rows, ['Column', 'Raw Profile', 'Target Type', 'Non-Null', 'Nulls', 'Examples'])))


In [ ]:
malformed_dates = []
invalid_response = []
range_results = {name: {'negative': 0, 'fractional': 0, 'min': None, 'max': None} for name in TARGETED_RANGE_COLUMNS}
category_columns = ['State', 'Soil_Type']
category_collisions = {}
hr_stats = {'negative': 0, 'fractional': 0, 'min': None, 'max': None}

for column_name in category_columns:
    variants = defaultdict(set)
    for row in rows:
        value = clean(row[column_name])
        if value is not None:
            variants[value.casefold()].add(value)
    category_collisions[column_name] = {key: sorted(values) for key, values in variants.items() if len(values) > 1}

for line_number, row in enumerate(rows, start=2):
    date_value = clean(row['Date'])
    if date_value is not None and not is_date(date_value):
        malformed_dates.append((line_number, date_value))
    response_value = clean(row['Response_Time_Min'])
    if response_value is not None and parse_decimal(response_value) is None:
        invalid_response.append((line_number, response_value))
    for column_name in TARGETED_RANGE_COLUMNS:
        numeric_value = parse_decimal(row[column_name])
        if numeric_value is None:
            continue
        if numeric_value < 0:
            range_results[column_name]['negative'] += 1
        if numeric_value != numeric_value.to_integral_value():
            range_results[column_name]['fractional'] += 1
        as_int = int(numeric_value)
        current_min = range_results[column_name]['min']
        current_max = range_results[column_name]['max']
        range_results[column_name]['min'] = as_int if current_min is None or as_int < current_min else current_min
        range_results[column_name]['max'] = as_int if current_max is None or as_int > current_max else current_max
    hr_value = parse_decimal(row['Human_Resources_Deployed'])
    if hr_value is not None:
        if hr_value < 0:
            hr_stats['negative'] += 1
        if hr_value != hr_value.to_integral_value():
            hr_stats['fractional'] += 1
        as_int = int(hr_value)
        hr_stats['min'] = as_int if hr_stats['min'] is None or as_int < hr_stats['min'] else hr_stats['min']
        hr_stats['max'] = as_int if hr_stats['max'] is None or as_int > hr_stats['max'] else hr_stats['max']

findings = [
    {'Check': '`Date` malformed values', 'Result': len(malformed_dates), 'Detail': 'None' if not malformed_dates else str(malformed_dates[:5])},
    {'Check': '`Response_Time_Min` malformed numeric values', 'Result': len(invalid_response), 'Detail': 'None' if not invalid_response else str(invalid_response[:5])},
    {'Check': '`State` category collisions', 'Result': len(category_collisions['State']), 'Detail': 'None' if not category_collisions['State'] else str(category_collisions['State'])},
    {'Check': '`Soil_Type` category collisions', 'Result': len(category_collisions['Soil_Type']), 'Detail': 'None' if not category_collisions['Soil_Type'] else str(category_collisions['Soil_Type'])},
    {'Check': '`Human_Resources_Deployed` interpretation', 'Result': 'numeric', 'Detail': f"min={hr_stats['min']}, max={hr_stats['max']}, negatives={hr_stats['negative']}, fractional={hr_stats['fractional']}"},
]
for column_name, metrics in range_results.items():
    findings.append({'Check': f'`{column_name}` range check', 'Result': 'valid', 'Detail': f"min={metrics['min']}, max={metrics['max']}, negatives={metrics['negative']}, fractional={metrics['fractional']}"})

display(Markdown(as_markdown_table(findings, ['Check', 'Result', 'Detail'])))


## Create the cleaned table without touching `raw_data`

The SQL below creates a new `cleaned_raw_data` table and loads cleaned values from `raw_data`. It only applies the requested scope: type conversion, whitespace trimming, and the targeted validation rules.


In [ ]:
def build_clean_table_sql(table_name=CLEAN_TABLE):
    column_defs = ['`raw_id` BIGINT UNSIGNED NOT NULL']
    for name, sql_type in COLUMN_SPECS:
        column_defs.append(f'`{name}` {sql_type} NULL')
    column_defs.extend([
        '`cleaned_at` TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP',
        'PRIMARY KEY (`raw_id`)',
        'KEY `idx_cleaned_raw_data_event_id` (`Event_ID`)',
        'KEY `idx_cleaned_raw_data_response_id` (`Response_ID`)',
        'KEY `idx_cleaned_raw_data_date` (`Date`)',
        'KEY `idx_cleaned_raw_data_state_district` (`State`, `District`)'
    ])
    return f"CREATE TABLE IF NOT EXISTS `{table_name}` (\n  " + ',\n  '.join(column_defs) + "\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;"

def clean_expression(column_name, sql_type):
    if column_name in {'Event_ID', 'State', 'District', 'Season', 'Soil_Type', 'Land_Use', 'Landslide_Occurred', 'Landslide_Risk', 'Response_ID', 'Compensation_Provided', 'Water_Supply_Disrupted', 'Communication_Disruption'}:
        return f"NULLIF(TRIM(`{column_name}`), '')"
    if column_name == 'Date':
        return '`Date`'
    if column_name == 'Response_Time_Min':
        return "CASE WHEN `Response_Time_Min` IS NULL OR `Response_Time_Min` < 0 THEN NULL ELSE CAST(`Response_Time_Min` AS DECIMAL(10,2)) END"
    if column_name in {'Historical_Landslide_Count', 'Houses_Damaged', 'Bridges_Damaged', 'Human_Resources_Deployed'}:
        return f"CASE WHEN `{column_name}` IS NULL OR `{column_name}` < 0 THEN NULL ELSE CAST(`{column_name}` AS SIGNED) END"
    if sql_type.startswith('INT'):
        return f"CAST(`{column_name}` AS SIGNED)"
    if sql_type.startswith('DECIMAL'):
        return f"CAST(`{column_name}` AS {sql_type})"
    return f'`{column_name}`'

create_sql = build_clean_table_sql()
insert_columns = ['raw_id'] + [name for name, _ in COLUMN_SPECS]
select_expressions = ['`raw_id`'] + [clean_expression(name, sql_type) for name, sql_type in COLUMN_SPECS]
insert_sql = f"INSERT INTO `{CLEAN_TABLE}` ({', '.join(f'`{column}`' for column in insert_columns)}) SELECT {', '.join(select_expressions)} FROM `{RAW_TABLE}`;"

print(create_sql)
print()
print(insert_sql)


In [ ]:
connection = pymysql.connect(
    host=os.getenv('MYSQL_HOST', 'localhost'),
    port=int(os.getenv('MYSQL_PORT', '3306')),
    user=os.getenv('MYSQL_USER', 'root'),
    password=os.getenv('MYSQL_PASSWORD', ''),
    database=os.getenv('MYSQL_DATABASE', 'landslide_analysis'),
    charset=os.getenv('MYSQL_CHARSET', 'utf8mb4'),
    autocommit=False,
)

with connection.cursor() as cursor:
    cursor.execute(create_sql)
    cursor.execute(f'TRUNCATE TABLE `{CLEAN_TABLE}`')
    cursor.execute(insert_sql)
    connection.commit()

with connection.cursor() as cursor:
    cursor.execute(f'SELECT COUNT(*) FROM `{RAW_TABLE}`')
    raw_count = cursor.fetchone()[0]
    cursor.execute(f'SELECT COUNT(*) FROM `{CLEAN_TABLE}`')
    clean_count = cursor.fetchone()[0]

connection.close()
display(Markdown(f'- Raw rows: **{raw_count:,}**\n- Cleaned rows: **{clean_count:,}**'))


In [ ]:
validation_queries = [
    ('Cleaned null Date count', f'SELECT COUNT(*) FROM `{CLEAN_TABLE}` WHERE `Date` IS NULL'),
    ('Cleaned negative Historical_Landslide_Count', f'SELECT COUNT(*) FROM `{CLEAN_TABLE}` WHERE `Historical_Landslide_Count` < 0'),
    ('Cleaned negative Houses_Damaged', f'SELECT COUNT(*) FROM `{CLEAN_TABLE}` WHERE `Houses_Damaged` < 0'),
    ('Cleaned negative Bridges_Damaged', f'SELECT COUNT(*) FROM `{CLEAN_TABLE}` WHERE `Bridges_Damaged` < 0'),
    ('Cleaned negative Response_Time_Min', f'SELECT COUNT(*) FROM `{CLEAN_TABLE}` WHERE `Response_Time_Min` < 0'),
]

connection = pymysql.connect(
    host=os.getenv('MYSQL_HOST', 'localhost'),
    port=int(os.getenv('MYSQL_PORT', '3306')),
    user=os.getenv('MYSQL_USER', 'root'),
    password=os.getenv('MYSQL_PASSWORD', ''),
    database=os.getenv('MYSQL_DATABASE', 'landslide_analysis'),
    charset=os.getenv('MYSQL_CHARSET', 'utf8mb4'),
)

results = []
with connection.cursor() as cursor:
    for label, query in validation_queries:
        cursor.execute(query)
        results.append({'Check': label, 'Count': cursor.fetchone()[0]})
connection.close()

display(Markdown(as_markdown_table(results, ['Check', 'Count'])))
